In [16]:
####------Section 1: Cleaning the raw data--------###

import pandas as pd
import re

# -----------------------------
# 1. Load your dataset
# -----------------------------
# Replace with your file path
df = pd.read_csv(r"data\address_with_house_number.csv", encoding="latin1")

# Assuming your column name is 'address'
# If different, change here
col = "address"

# -----------------------------
# 2. Dictionary for standardization
# -----------------------------
replacements = {
    r'\bmarga\b': 'marg',
    r'\bmrag\b': 'marg',
    r'\bmargh\b': 'marg',
    r'\broad\b': 'marg',
    r'\brd\b': 'marg',
    r'\bbnesh?wor\b': 'baneshwor',
    r'\bbns\b': 'baneshwor',
    r'\bktm\b': 'kathmandu',
    r'\bnew banes?wor\b': 'new baneshwor',
    r'\bold banes?wor\b': 'old baneshwor',
    r'\bmid banes?wor\b': 'mid baneshwor',
    r'\bbattisputali\b': 'battisputali',
    r'\bnaxal\b': 'naxal',
}

# -----------------------------
# 3. Cleaning function
# -----------------------------
def clean_address(text):
    if pd.isna(text):
        return ""

    # Lowercase
    text = text.lower()

    # Remove phone numbers
    text = re.sub(r'\b\d{7,}\b', '', text)

    # Remove Nepali unicode noise (optional)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # Remove unwanted phrases
    noise_words = [
        "before", "after", "only", "please", "call", 
        "delivery", "urgent", "bag", "pic"
    ]
    for word in noise_words:
        text = re.sub(rf'\b{word}\b.*', '', text)

    # Apply replacements
    for pattern, repl in replacements.items():
        text = re.sub(pattern, repl, text)

    # Normalize "near", "opp", etc.
    text = re.sub(r'\bnear\b', 'near', text)
    text = re.sub(r'\bopp\b|\bopposite\b', 'opposite', text)

    # Remove extra symbols
    text = re.sub(r'[^\w\s,/-]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Capitalize nicely
    text = text.title()

    return text

# -----------------------------
# 4. Apply cleaning
# -----------------------------
df["cleaned_address"] = df[col].apply(clean_address)

# -----------------------------
# 5. Save output
# -----------------------------
df.to_csv(r"data\1Clean_Address.csv", index=False)

print("✅ Cleaning complete. File saved as cleaned_addresses.csv")

✅ Cleaning complete. File saved as cleaned_addresses.csv


In [17]:
####-----Section 2: Separate Marg and Galli from the excel file (Separating Street)------#####

import pandas as pd
import re

# Load your file
df = pd.read_csv(r"data\1Clean_Address.csv")

# Function to extract Marg/Galli
def extract_street(address):
    if pd.isna(address):
        return None
    
    # Pattern to capture words ending with Marg or Galli
    match = re.search(r'([A-Za-z\s]+?\b(?:Marg|Galli))', address)
    
    if match:
        return match.group(1).strip()
    else:
        return None

# Apply function
df["street_name"] = df["cleaned_address"].apply(extract_street)

# Save file
df.to_csv(r"data\2street_address.csv", index=False)

print("✅ Street extraction complete!")

✅ Street extraction complete!


In [18]:
####-------Section 3: House Number extraction from the field.-----#####

import pandas as pd
import re

# -----------------------------
# Load your file
# -----------------------------
df = pd.read_csv(
    r"data\2street_address.csv",
    encoding="latin1"
)

# -----------------------------
# Function to extract house number
# -----------------------------
def extract_house_number(address):
    if pd.isna(address):
        return None
    
    # Extract ONLY from beginning (avoids "200 meters" issue)
    match = re.search(r'^\s*(\d+\/\d+|\d+)', address)
    
    
    if match:
        return match.group(1)
    else:
        return None

# -----------------------------
# Apply extraction
# -----------------------------
df["house_number"] = df["cleaned_address"].apply(extract_house_number)

# -----------------------------
# Force as text (IMPORTANT)
# -----------------------------
df["house_number"] = df["house_number"].astype(str)

# Optional: remove "None" string if created
df["house_number"] = df["house_number"].replace("None", "")

# -----------------------------
# Save as Excel (BEST PRACTICE)
# -----------------------------

output_path = r"data\3_housenumber_street_address.csv"
df.to_csv(output_path, index=False)

print("✅ House number extraction complete! Saved as Excel without date issue.")

✅ House number extraction complete! Saved as Excel without date issue.


In [20]:
#####--------Section 4: Geocode according to house number.-----#####

#Kam garyo for x/y.

import geopandas as gpd
import pandas as pd
import re
from shapely.geometry import Point
import numpy as np

# ----------------------------
# Load data
# ----------------------------
roads = gpd.read_file(r"data\road.shp")
df = pd.read_csv(r"data\3_housenumber_street_address.csv")

# Ensure CRS is projected (meters)
roads = roads.to_crs(epsg=32645)  # UTM zone for Kathmandu

# ----------------------------
# Clean road names
# ----------------------------
roads["road_name"] = roads["kvmp_nam"].str.strip().str.lower()
df["street_name_clean"] = df["street_name"].str.strip().str.lower()

# ----------------------------
# Function to parse house number
# ----------------------------
def parse_house_number(hn):
    if pd.isna(hn):
        return None, None
    
    match = re.match(r"(\d+)\/(\d+)", str(hn))
    
    if match:
        return int(match.group(1)), int(match.group(2))
    
    elif str(hn).isdigit():
        return int(hn), None
    
    return None, None

# ----------------------------
# Function to get point
# ----------------------------
def get_point(row):

    road_name = row["street_name_clean"]
    hn = row["house_number"]

    x_dist, y_offset = parse_house_number(hn)

    if x_dist is None:
        return None

    # ----------------------------
    # Match road
    # ----------------------------
    road_match = roads[roads["road_name"] == road_name]

    if road_match.empty:
        return None

    road_geom = road_match.geometry.values[0]

    # Handle multilines
    if road_geom.geom_type == "MultiLineString":
        road_geom = max(road_geom.geoms, key=lambda x: x.length)

    # ----------------------------
    # Step 1: move along road (x)
    # ----------------------------
    x_dist = min(x_dist, road_geom.length)
    point_on_line = road_geom.interpolate(x_dist)

    # If no y (only single number)
    if y_offset is None:
        return point_on_line

    # ----------------------------
    # Step 2: get direction of road
    # ----------------------------
    d = road_geom.project(point_on_line)

    delta = 0.5  # small step for direction
    p1 = road_geom.interpolate(max(0, d - delta))
    p2 = road_geom.interpolate(min(road_geom.length, d + delta))

    x1, y1 = p1.x, p1.y
    x2, y2 = p2.x, p2.y

    px, py = point_on_line.x, point_on_line.y

    # Direction vector
    dx = x2 - x1
    dy = y2 - y1

    length = np.sqrt(dx**2 + dy**2)
    if length == 0:
        return point_on_line

    ux = dx / length
    uy = dy / length

    # ----------------------------
    # Step 3: left/right using odd-even
    # ----------------------------
    if y_offset % 2 == 1:
        side = "Left"
    else:
        side = "Right"

    # Perpendicular offset
    if side == "Left":
        ox = -uy * y_offset
        oy = ux * y_offset
    else:
        ox = uy * y_offset
        oy = -ux * y_offset

    final_point = Point(px + ox, py + oy)

    return final_point

# ----------------------------
# Apply function
# ----------------------------
df["geometry"] = df.apply(get_point, axis=1)

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=roads.crs)

# Convert back to lat/lon
gdf = gdf.to_crs(epsg=4326)

# Extract lat/lon
gdf["latitude"] = gdf.geometry.y
gdf["longitude"] = gdf.geometry.x

# ----------------------------
# Save output
# ----------------------------
gdf.drop(columns="geometry").to_csv(
    r"data\Final Output with Lat_Long.csv",
    index=False
)

print("✅ Address-based coordinate generation complete!")

✅ Address-based coordinate generation complete!


C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D LineString' is converted to 'LineString Z'
  return ogr_read(
